### Checking dimensions 

In [ ]:
# 1) Make sure the CRAN-installed one isn’t attached
if ("package:exdqlm" %in% search()) detach("package:exdqlm", unload = TRUE)

# 2) Load your local repo build (this compiles/loads the source in /home/antonio/code/exdqlm)
install.packages("devtools")        # if needed, once
devtools::load_all("/home/antonio/code/exdqlm")
# You should see: Loading exdqlm

# 3) Verify we’re using the local one
getFromNamespace("exdqlmISVB", "exdqlm")  # should point to your modified function


In [ ]:
# Data
data("BTflow", package = "exdqlm")
data("nino34", package = "exdqlm")

# Components
trend.comp <- polytrendMod(order = 1, m0 = 3, C0 = 0.1)
seas.comp  <- seasMod(p = 12, h = 1, C0 = diag(1, 2))
base.mod   <- combineMods(trend.comp, seas.comp)

reg.comp <- list(m0 = 0, C0 = 1, FF = matrix(nino34, nrow = 1), GG = 1)
model.w.reg <- combineMods(base.mod, reg.comp)

df_vec  <- c(1, 0.9, 0.95)
dim_vec <- c(1,   2,    1)

set.seed(1)
fit <- exdqlmISVB(
  y        = log(BTflow),
  p0       = 0.5,
  model    = model.w.reg,
  df       = df_vec,
  dim.df   = dim_vec,
  sig.init = 0.1,
  gam.init = -0.1,
  tol      = 0.05,
  verbose  = FALSE,
  debug_shapes = TRUE,    # <— turn on the prints
  debug_every  = 1        # print every iteration (use 10 if too chatty)
)


### Testing C++ Implementation

In [ ]:
# 0) Point at your package root (where DESCRIPTION lives)
pkg <- "~/code/exdqlm"   # adjust to your actual path
stopifnot(file.exists(file.path(pkg, "DESCRIPTION")))

# 1) If the package is already loaded, unload it
if ("exdqlm" %in% loadedNamespaces()) {
  detach("package:exdqlm", unload = TRUE, character.only = TRUE)
  unloadNamespace("exdqlm")
}

# 2) Regenerate Rcpp glue
Rcpp::compileAttributes(pkgdir = pkg)

# 3) Sync roxygen docs/NAMESPACE
devtools::document(pkg)

# 4) Load your development package (from source, not installed CRAN version)
devtools::load_all(pkg)

# 5) Sanity check
exists("update_theta_cpp")     # should be TRUE


In [ ]:
## --- ELBO tests for exdqlmISVB ---

options(
  exdqlm.use_cpp_kf  = TRUE,   # start with C++ KF
  exdqlm.compute_elbo = TRUE,  # turn ELBO on
  exdqlm.tol_elbo     = 1e-6   # small so stopping is driven mainly by `tol`
)

# package datasets & helpers (loaded via load_all)
data("BTflow")
data("nino34")

trend.comp <- polytrendMod(order = 1, m0 = 3, C0 = 0.1)
seas.comp  <- seasMod(p = 12, h = 1, C0 = diag(1, 2))
base.mod   <- combineMods(trend.comp, seas.comp)

reg.comp <- list(m0 = 0, C0 = 1, FF = matrix(nino34, nrow = 1), GG = 1)
model.w.reg <- combineMods(base.mod, reg.comp)

df_vec  <- c(1, 0.9, 0.95)
dim_vec <- c(1,   2,  1)

# ---------- Helpers ----------
theta_entropy_recalc <- function(th) {
  p  <- nrow(th$sm)
  TT <- dim(th$sC)[3]
  0.5 * sum(vapply(seq_len(TT), function(t) {
    p * (1 + log(2*pi)) + determinant(th$sC[,,t], logarithm = TRUE)$modulus[1]
  }, numeric(1)))
}

monotone_up_to <- function(x, tol_drop = 1e-4) {
  # allow tiny downward blips from IS noise
  if (length(x) < 2) return(TRUE)
  min(diff(x)) >= -abs(tol_drop)
}

last_elbo <- function(fit) {
  if (!is.null(fit$diagnostics$elbo)) tail(fit$diagnostics$elbo, 1)[[1]] else NA_real_
}

# ---------- Fit: R backend ----------
options(exdqlm.use_cpp_kf = FALSE)
set.seed(1)
fit_R <- exdqlmISVB(
  y        = log(BTflow),
  p0       = 0.5,
  model    = model.w.reg,
  df       = df_vec,
  dim.df   = dim_vec,
  sig.init = 0.1,
  gam.init = -0.1,
  tol      = 0.05,
  verbose  = TRUE
)

# ---------- Fit: C++ backend ----------
options(exdqlm.use_cpp_kf = TRUE)
set.seed(1)
fit_C <- exdqlmISVB(
  y        = log(BTflow),
  p0       = 0.5,
  model    = model.w.reg,
  df       = df_vec,
  dim.df   = dim_vec,
  sig.init = 0.1,
  gam.init = -0.1,
  tol      = 0.05,
  verbose  = TRUE
)

# ---------- Core KF equality (as before) ----------
stopifnot(isTRUE(all.equal(fit_R$theta.out$exps,  fit_C$theta.out$exps,  tol = 1e-6, check.attributes = FALSE)))
stopifnot(isTRUE(all.equal(fit_R$theta.out$vars,  fit_C$theta.out$vars,  tol = 1e-6, check.attributes = FALSE)))
stopifnot(isTRUE(all.equal(fit_R$theta.out$exps2, fit_C$theta.out$exps2, tol = 1e-6, check.attributes = FALSE)))

# ---------- ELBO presence ----------
stopifnot(!is.null(fit_R$diagnostics$elbo), !is.null(fit_C$diagnostics$elbo))

# ---------- ELBO monotonicity (allow tiny dips) ----------
stopifnot(monotone_up_to(fit_R$diagnostics$elbo, tol_drop = 1e-3))
stopifnot(monotone_up_to(fit_C$diagnostics$elbo, tol_drop = 1e-3))

# ---------- θ-entropy check (C++ bridge exposes it; R path may not) ----------
if (!is.null(fit_C$theta.out$elbo_theta)) {
  etheta_C <- fit_C$theta.out$elbo_theta
  etheta_C_re <- theta_entropy_recalc(fit_C$theta.out)
  stopifnot(isTRUE(all.equal(etheta_C, etheta_C_re, tol = 1e-8)))
}

# ---------- Final ELBO comparability (R vs C++) ----------
final_R <- last_elbo(fit_R)
final_C <- last_elbo(fit_C)
print(sprintf("Final ELBO — R: %.6f | C++: %.6f | diff: %.3e", final_R, final_C, final_C - final_R))

# Allow a modest tolerance because IS weights differ slightly run-to-run
stopifnot(isTRUE(all.equal(final_R, final_C, tol = 1e-2)))

# ---------- Optional: report sigma–gamma ELBO piece if available ----------
if (!is.null(fit_C$gammasig.out$elbo_logZ)) {
  cat(sprintf("Sigma–Gamma elbo_logZ (C++ run): %.6f\n", fit_C$gammasig.out$elbo_logZ))
}
if (!is.null(fit_R$gammasig.out$elbo_logZ)) {
  cat(sprintf("Sigma–Gamma elbo_logZ (R run): %.6f\n", fit_R$gammasig.out$elbo_logZ))
}

cat("ELBO tests passed ✓\n")


### Sampling Check

In [ ]:
pkg <- "~/code/exdqlm"           # <- adjust if needed
stopifnot(file.exists(file.path(pkg, "DESCRIPTION")))

if ("exdqlm" %in% loadedNamespaces()) {
  try(detach("package:exdqlm", unload = TRUE, character.only = TRUE), silent = TRUE)
  try(unloadNamespace("exdqlm"), silent = TRUE)
}

Rcpp::compileAttributes(pkgdir = pkg)
devtools::document(pkg)
devtools::load_all(pkg, quiet = TRUE)

# sanity: core C++ symbol present
stopifnot(exists("update_theta_cpp", mode = "function"))

In [ ]:
options(
  exdqlm.use_cpp_kf       = TRUE,   # C++ Kalman bridge
  exdqlm.use_cpp_samplers = FALSE,  # R samplers for now
  exdqlm.use_cpp_postpred = FALSE,
  exdqlm.compute_elbo     = TRUE,
  exdqlm.tol_elbo         = 1e-6
)

data("BTflow", package = "exdqlm")
y <- log(BTflow)  # small slice for speed

trend.comp <- polytrendMod(order = 1, m0 = 0, C0 = 1)
seas.comp  <- seasMod(p = 12, h = 1, C0 = diag(1, 2))
mod        <- combineMods(trend.comp, seas.comp)

set.seed(123)
fit_Rsamplers <- exdqlmISVB(
  y        = y,
  p0       = 0.5,
  model    = mod,
  df       = c(1, 0.98),
  dim.df   = c(1, 2),
  sig.init = 0.2,
  gam.init = -0.1,
  tol      = 0.05,
  verbose  = TRUE
)

# ELBO sanity: weakly monotone (allow tiny IS blips)
stopifnot(length(fit_Rsamplers$diagnostics$elbo) > 1L)
stopifnot(min(diff(fit_Rsamplers$diagnostics$elbo)) >= -1e-3)

# Shape checks
p  <- nrow(fit_Rsamplers$theta.out$sm)
TT <- ncol(fit_Rsamplers$theta.out$sm)
ns <- dim(fit_Rsamplers$samp.theta)[3]

stopifnot(identical(dim(fit_Rsamplers$samp.theta),    c(p, TT, ns)))
stopifnot(identical(dim(fit_Rsamplers$samp.post.pred), c(TT, ns)))
stopifnot(is.finite(mean(fit_Rsamplers$samp.post.pred)))

# Quick numeric peek
c(
  mean_theta = mean(fit_Rsamplers$samp.theta),
  mean_pred  = mean(fit_Rsamplers$samp.post.pred),
  mean_sigma = mean(fit_Rsamplers$samp.sigma),
  mean_gamma = mean(fit_Rsamplers$samp.gamma)
)

In [ ]:
options(exdqlm.use_cpp_samplers = TRUE)

set.seed(123)
fit_CPPsamplers <- exdqlmISVB(
  y        = y,
  p0       = 0.5,
  model    = mod,
  df       = c(1, 0.98),
  dim.df   = c(1, 2),
  sig.init = 0.2,
  gam.init = -0.1,
  tol      = 0.05,
  verbose  = TRUE
)

# Same shape checks
p2  <- nrow(fit_CPPsamplers$theta.out$sm)
TT2 <- ncol(fit_CPPsamplers$theta.out$sm)
ns2 <- dim(fit_CPPsamplers$samp.theta)[3]
stopifnot(identical(c(p2,TT2), c(p,TT)))

stopifnot(identical(dim(fit_CPPsamplers$samp.theta),     c(p, TT, ns2)))
stopifnot(identical(dim(fit_CPPsamplers$samp.post.pred), c(TT, ns2)))
stopifnot(is.finite(mean(fit_CPPsamplers$samp.post.pred)))

# ELBO endpoints (should be close, not necessarily identical)
c(
  final_Rsamplers  = tail(fit_Rsamplers$diagnostics$elbo, 1),
  final_CPPsamplers = tail(fit_CPPsamplers$diagnostics$elbo, 1),
  diff = tail(fit_CPPsamplers$diagnostics$elbo, 1) - tail(fit_Rsamplers$diagnostics$elbo, 1)
)


In [ ]:
options(exdqlm.use_cpp_samplers = FALSE)  # sampler choice irrelevant for KF parity

# R KF
options(exdqlm.use_cpp_kf = FALSE)
set.seed(1)
fit_RKF <- exdqlmISVB(
  y        = y,
  p0       = 0.5,
  model    = mod,
  df       = c(1, 0.98),
  dim.df   = c(1, 2),
  sig.init = 0.2,
  gam.init = -0.1,
  tol      = 0.05,
  verbose  = FALSE
)

# C++ KF
options(exdqlm.use_cpp_kf = TRUE)
set.seed(1)
fit_CKF <- exdqlmISVB(
  y        = y,
  p0       = 0.5,
  model    = mod,
  df       = c(1, 0.98),
  dim.df   = c(1, 2),
  sig.init = 0.2,
  gam.init = -0.1,
  tol      = 0.05,
  verbose  = FALSE
)

stopifnot(isTRUE(all.equal(fit_RKF$theta.out$exps,  fit_CKF$theta.out$exps,  tol = 1e-6, check.attributes = FALSE)))
stopifnot(isTRUE(all.equal(fit_RKF$theta.out$vars,  fit_CKF$theta.out$vars,  tol = 1e-6, check.attributes = FALSE)))
stopifnot(isTRUE(all.equal(fit_RKF$theta.out$exps2, fit_CKF$theta.out$exps2, tol = 1e-6, check.attributes = FALSE)))


### Testing LD Implementation

In [ ]:
## ── 0) Point at your package root (where DESCRIPTION lives)
pkg <- "~/code/exdqlm"   # adjust
stopifnot(file.exists(file.path(pkg, "DESCRIPTION")))

## ── 1) If already loaded, unload
if ("exdqlm" %in% loadedNamespaces()) {
  detach("package:exdqlm", unload = TRUE, character.only = TRUE)
  unloadNamespace("exdqlm")
}

## ── 2) Regenerate Rcpp glue
Rcpp::compileAttributes(pkgdir = pkg)

## ── 3) Sync roxygen docs/NAMESPACE
devtools::document(pkg)

## ── 4) Load dev package
devtools::load_all(pkg)

## ── 5) Sanity check C++ KF symbol
has_cpp_bridge <- exists("update_theta_bridge")
has_cpp_cpp    <- exists("update_theta_cpp")
stopifnot(has_cpp_bridge || has_cpp_cpp)

## ── ELBO options (common)
options(
  exdqlm.use_cpp_kf   = TRUE,  # we will toggle to FALSE for R-KF comparisons
  exdqlm.compute_elbo = TRUE,
  exdqlm.tol_elbo     = 1e-6,
  exdqlm.use_cpp_samplers = FALSE,   # keep samplers off; we’re only testing KF+ELBO
  exdqlm.use_cpp_postpred = FALSE
)

## ── Data & model
data("BTflow")
data("nino34")

trend.comp <- polytrendMod(order = 1, m0 = 3, C0 = 0.1)
seas.comp  <- seasMod(p = 12, h = 1, C0 = diag(1, 2))
base.mod   <- combineMods(trend.comp, seas.comp)

# nino34 as a time-varying regressor (one-dim)
reg.comp <- list(m0 = 0, C0 = 1, FF = matrix(nino34, nrow = 1), GG = 1)
model.w.reg <- combineMods(base.mod, reg.comp)

df_vec  <- c(1, 0.9, 0.95)
dim_vec <- c(1,   2,  1)

## ── Helpers
theta_entropy_recalc <- function(th) {
  p  <- nrow(th$sm)
  TT <- dim(th$sC)[3]
  0.5 * sum(vapply(seq_len(TT), function(t) {
    p * (1 + log(2*pi)) + determinant(th$sC[,,t], logarithm = TRUE)$modulus[1]
  }, numeric(1)))
}

monotone_up_to <- function(x, tol_drop = 1e-4) {
  if (length(x) < 2) return(TRUE)
  min(diff(x)) >= -abs(tol_drop)
}

last_elbo <- function(fit) {
  if (!is.null(fit$diagnostics$elbo)) tail(fit$diagnostics$elbo, 1)[[1]] else NA_real_
}

gs_sanity <- function(fit, p0) {
  L <- L.fn(p0); U <- U.fn(p0)
  g  <- tail(fit$seq.gamma, 1)
  s  <- tail(fit$seq.sigma, 1)
  stopifnot(is.finite(g), is.finite(s), s > 0, g >= L - 1e-8, g <= U + 1e-8)
  invisible(list(gamma_final = g, sigma_final = s, L = L, U = U))
}

print_elbos <- function(tag, Rfit, Cfit, tol = 1e-2) {
  eR <- last_elbo(Rfit); eC <- last_elbo(Cfit)
  cat(sprintf("[%s] final ELBO — R: %.6f | C++: %.6f | diff: %.3e\n", tag, eR, eC, eC - eR))
  stopifnot(isTRUE(all.equal(eR, eC, tol = tol)))
}

check_theta_equality <- function(Rfit, Cfit) {
  stopifnot(
    isTRUE(all.equal(Rfit$theta.out$exps,  Cfit$theta.out$exps,  tol = 1e-6, check.attributes = FALSE)),
    isTRUE(all.equal(Rfit$theta.out$vars,  Cfit$theta.out$vars,  tol = 1e-6, check.attributes = FALSE)),
    isTRUE(all.equal(Rfit$theta.out$exps2, Cfit$theta.out$exps2, tol = 1e-6, check.attributes = FALSE))
  )
}

check_theta_entropy <- function(Cfit) {
  if (!is.null(Cfit$theta.out$elbo_theta)) {
    etheta_C    <- Cfit$theta.out$elbo_theta
    etheta_re   <- theta_entropy_recalc(Cfit$theta.out)
    stopifnot(isTRUE(all.equal(etheta_C, etheta_re, tol = 1e-8)))
  }
}

## ── Common spec
y_vec   <- log(BTflow)
p0      <- 0.9
sig0    <- 0.01
gam0    <- 0
tol_val <- 0.05

## ── A) ISVB: R-KF vs C++-KF (algorithm should match)
set.seed(1)
options(exdqlm.use_cpp_kf = FALSE)
fit_IS_R <- exdqlmISVB(
  y        = y_vec,
  p0       = p0,
  model    = model.w.reg,
  df       = df_vec,
  dim.df   = dim_vec,
  sig.init = sig0,
  gam.init = gam0,
  tol      = tol_val,
  verbose  = TRUE
)

set.seed(1)
options(exdqlm.use_cpp_kf = TRUE)
fit_IS_C <- exdqlmISVB(
  y        = y_vec,
  p0       = p0,
  model    = model.w.reg,
  df       = df_vec,
  dim.df   = dim_vec,
  sig.init = sig0,
  gam.init = gam0,
  tol      = tol_val,
  verbose  = TRUE
)

# KF equality
check_theta_equality(fit_IS_R, fit_IS_C)

# ELBO presence + near-monotonicity
stopifnot(!is.null(fit_IS_R$diagnostics$elbo), !is.null(fit_IS_C$diagnostics$elbo))
stopifnot(monotone_up_to(fit_IS_R$diagnostics$elbo, tol_drop = 1e-3))
stopifnot(monotone_up_to(fit_IS_C$diagnostics$elbo, tol_drop = 1e-3))

# θ-entropy agreement (only if exposed)
check_theta_entropy(fit_IS_C)

# Final ELBO comparability
print_elbos("ISVB", fit_IS_R, fit_IS_C, tol = 1e-2)

# σ/γ sanity
invisible(gs_sanity(fit_IS_R, p0))
invisible(gs_sanity(fit_IS_C, p0))


## ── B) LDVB: R-KF vs C++-KF (same LD objective, KF should match)
set.seed(2)
options(exdqlm.use_cpp_kf = FALSE)
fit_LD_R <- exdqlmLDVB(
  y        = y_vec,
  p0       = p0,
  model    = model.w.reg,
  df       = df_vec,
  dim.df   = dim_vec,
  sig.init = sig0,
  gam.init = gam0,
  tol      = tol_val,
  verbose  = TRUE
)

set.seed(2)
options(exdqlm.use_cpp_kf = TRUE)
fit_LD_C <- exdqlmLDVB(
  y        = y_vec,
  p0       = p0,
  model    = model.w.reg,
  df       = df_vec,
  dim.df   = dim_vec,
  sig.init = sig0,
  gam.init = gam0,
  tol      = tol_val,
  verbose  = TRUE
)

# KF equality
check_theta_equality(fit_LD_R, fit_LD_C)

# ELBO presence + near-monotonicity
stopifnot(!is.null(fit_LD_R$diagnostics$elbo), !is.null(fit_LD_C$diagnostics$elbo))
stopifnot(monotone_up_to(fit_LD_R$diagnostics$elbo, tol_drop = 1e-3))
stopifnot(monotone_up_to(fit_LD_C$diagnostics$elbo, tol_drop = 1e-3))

# θ-entropy agreement (only if exposed)
check_theta_entropy(fit_LD_C)

# Final ELBO comparability
print_elbos("LDVB", fit_LD_R, fit_LD_C, tol = 1e-2)

# σ/γ sanity
invisible(gs_sanity(fit_LD_R, p0))
invisible(gs_sanity(fit_LD_C, p0))

## ── C) Optional: quick σ/γ trajectory glance
cat(sprintf(
  "ISVB:  last sigma=%.4g, last gamma=%.4g | LDVB: last sigma=%.4g, last gamma=%.4g\n",
  tail(fit_IS_C$seq.sigma, 1), tail(fit_IS_C$seq.gamma, 1),
  tail(fit_LD_C$seq.sigma, 1), tail(fit_LD_C$seq.gamma, 1)
))

cat("All ISVB/LDVB C++ vs R checks passed ✓\n")


In [ ]:
## ---- Comparisons (safe, no stale names) ----

# (1) R vs C++ within each algorithm
check_theta_equality(fit_IS_R, fit_IS_C)
check_theta_entropy(fit_IS_C)
print_elbos("ISVB", fit_IS_R, fit_IS_C, tol = 1e-2)

check_theta_equality(fit_LD_R, fit_LD_C)
check_theta_entropy(fit_LD_C)
print_elbos("LDVB", fit_LD_R, fit_LD_C, tol = 1e-2)

# (2) ISVB vs LDVB on the SAME backend (choose C++ here)
cmp_elbo <- function(fit1, fit2, name1="ISVB", name2="LDVB") {
  e1 <- tail(fit1$diagnostics$elbo, 1)
  e2 <- tail(fit2$diagnostics$elbo, 1)
  cat(sprintf("%s ELBO: %.6f | %s ELBO: %.6f | diff: %.3e\n", name1, e1, name2, e2, e2 - e1))
}

cmp_fit <- function(fit1, fit2, name1="ISVB", name2="LDVB") {
  s1 <- tail(fit1$map.standard.forecast.errors, 200)
  s2 <- tail(fit2$map.standard.forecast.errors, 200)
  cat(sprintf("MAP std. err (last 200) — %s: mean=%.3f sd=%.3f | %s: mean=%.3f sd=%.3f\n",
              name1, mean(s1), sd(s1), name2, mean(s2), sd(s2)))
}

# Compare algorithms using the C++ KF fits:
cmp_elbo(fit_IS_C, fit_LD_C, "ISVB(C++)", "LDVB(C++)")
cmp_fit(fit_IS_C, fit_LD_C, "ISVB(C++)", "LDVB(C++)")

cat(sprintf(
  "ISVB(C++):  last sigma=%.4g, last gamma=%.4g | LDVB(C++): last sigma=%.4g, last gamma=%.4g\n",
  tail(fit_IS_C$seq.sigma, 1), tail(fit_IS_C$seq.gamma, 1),
  tail(fit_LD_C$seq.sigma, 1), tail(fit_LD_C$seq.gamma, 1)
))


In [ ]:
# Zoomed posterior predictive plot over an index window
pp_plot_zoom <- function(
  y, samples, p0,
  window = NULL,              # e.g. c(700, 1012); if NULL uses full range
  indices = NULL,             # alternative to 'window': explicit integer vector
  backtransform = identity,   # e.g. exp for original scale
  conf = c(0.025, 0.975),
  spaghetti_n = 0,            # number of sample paths to draw (0 = none)
  spaghetti_alpha = 0.2,
  col_band = "steelblue",
  col_q = "orange3",          # p0-quantile line color
  col_obs = "black",
  pch_obs = 16,
  cex_obs = 0.5,
  lwd_q = 3,                  # thickness of the p0-quantile line
  main = "Posterior predictive (zoom)",
  xlab = "index", ylab = "y"
) {
  stopifnot(is.numeric(y), is.matrix(samples), length(p0) == 1, p0 > 0, p0 < 1)
  TT <- length(y)

  # Ensure samples are TT x ns (transpose if needed)
  if (nrow(samples) != TT) {
    if (ncol(samples) == TT) samples <- t(samples)
    else stop("samples must be TT x ns or ns x TT.")
  }

  # Select indices
  if (!is.null(indices)) {
    idx <- intersect(indices, seq_len(TT))
  } else if (!is.null(window)) {
    idx <- seq.int(max(1, window[1]), min(TT, window[2]))
  } else {
    idx <- seq_len(TT)
  }
  if (length(idx) < 2) stop("Need at least 2 points to plot.")

  Y <- y[idx]
  S <- samples[idx, , drop = FALSE]

  # Quantiles (lo, p0, hi)
  probs <- sort(unique(c(conf[1], p0, conf[2])))
  Q <- if (requireNamespace("matrixStats", quietly = TRUE)) {
    matrixStats::rowQuantiles(S, probs = probs, na.rm = TRUE)
  } else {
    t(apply(S, 1, quantile, probs = probs, na.rm = TRUE))
  }
  il <- which.min(abs(probs - conf[1]))
  ih <- which.min(abs(probs - conf[2]))
  iq <- which.min(abs(probs - p0))

  # Backtransform everything
  Y_bt <- backtransform(Y)
  Q_lo <- backtransform(Q[, il])
  Q_hi <- backtransform(Q[, ih])
  Q_p0 <- backtransform(Q[, iq])

  # Axis
  x <- idx
  ylim <- range(c(Y_bt, Q_lo, Q_hi), finite = TRUE)

  op <- par(mar = c(3.5, 4.2, 3, 1) + 0.1)
  on.exit(par(op), add = TRUE)
  plot(x, Y_bt, type = "n", xlab = xlab, ylab = ylab, main = main, ylim = ylim)

  # 95% ribbon
  polygon(c(x, rev(x)), c(Q_lo, rev(Q_hi)),
          border = NA, col = adjustcolor(col_band, alpha.f = 0.30))

  # p0-quantile line
  lines(x, Q_p0, lwd = lwd_q, lty = 1, col = col_q)

  # spaghetti (subset for readability)
  if (spaghetti_n > 0) {
    ns <- ncol(S)
    keep <- if (spaghetti_n >= ns) seq_len(ns) else sample(ns, spaghetti_n)
    matlines(x, backtransform(S[, keep, drop = FALSE]),
             col = adjustcolor("grey20", spaghetti_alpha), lwd = 1)
  }

  # observed (points; flip to lines if you prefer)
  points(x, Y_bt, pch = pch_obs, cex = cex_obs, col = col_obs)

  # coverage inside ribbon
  cov95 <- mean(Y_bt >= Q_lo & Y_bt <= Q_hi, na.rm = TRUE)

  # legend label for p0
  p_lbl <- if (abs(p0 - 0.5) < 1e-8) "q0.5" else sprintf("q%.3g", p0)

  legend("topleft", bty = "n",
         legend = c("observed", p_lbl, "95% band",
                    if (spaghetti_n > 0) sprintf("samples (n=%d)", spaghetti_n) else NULL),
         pch    = c(16, NA, NA, NA),
         lty    = c(NA, 1, 1, 1),
         lwd    = c(NA, lwd_q, 8, 1),
         col    = c(col_obs, col_q, adjustcolor(col_band, 0.6), "grey20"))
  invisible(list(coverage_95 = cov95, idx = idx,
                 lo = Q_lo, q = Q_p0, hi = Q_hi))
}


In [ ]:
# ISVB (C++)
pp_plot_zoom(
  y = y_vec,
  samples = fit_IS_C$samp.post.pred,
  p0 = p0,                          # <- use your model's p0
  window = c(1012 - 50, 1012),
  backtransform = identity,
  spaghetti_n = 40,
  main = "ISVB (C++) — last 50 points"
)

# LDVB (C++)
pp_plot_zoom(
  y = y_vec,
  samples = fit_LD_C$samp.post.pred,
  p0 = p0,
  window = c(1012 - 50, 1012),
  backtransform = identity,
  spaghetti_n = 40,
  col_band = "tomato",
  main = "LDVB (C++) — last 50 points"
)


### --- Simulation ---

In [ ]:
# Required packages:
# install.packages(c("mvtnorm", "sn"))
library(mvtnorm)
library(sn)

# ------------------------- helpers -------------------------

# Make a matrix symmetric positive-definite with minimal jitter
.ensure_spd <- function(M, jitter = 1e-10) {
  S <- 0.5 * (M + t(M))  # symmetrize
  ev <- eigen(S, symmetric = TRUE, only.values = TRUE)$values
  min_ev <- min(ev)
  if (min_ev <= jitter) {
    S <- S + diag(jitter - min_ev + 1e-12, nrow(S))
  }
  S
}

# Robust broadcasting that preserves each time-slice exactly
.as_timevarying <- function(M, T, expected_shape, name) {
  if (is.null(M)) return(NULL)

  if (is.function(M)) {
    out <- array(NA_real_, dim = c(T, expected_shape))
    for (t in seq_len(T)) {
      Mt <- as.matrix(M(t - 1L))
      if (!identical(dim(Mt), expected_shape)) {
        stop(sprintf("%s(t) has shape (%s), expected (%s)",
                     name, paste(dim(Mt), collapse=","), paste(expected_shape, collapse=",")))
      }
      out[t, , ] <- Mt
    }
    return(out)
  }

  arr <- as.array(M)
  d <- dim(arr)

  if (length(d) == length(expected_shape)) {
    # constant matrix → copy to each t
    out <- array(NA_real_, dim = c(T, expected_shape))
    for (t in seq_len(T)) out[t, , ] <- arr
    return(out)
  } else if (length(d) == length(expected_shape) + 1L) {
    if (d[1] != T || !identical(d[-1], expected_shape)) {
      stop(sprintf("%s has shape (%s), expected (T, %s)",
                   name, paste(d, collapse=","), paste(expected_shape, collapse=",")))
    }
    return(arr)
  } else {
    stop(sprintf("%s has invalid ndim=%d", name, length(d)))
  }
}

.peek_shape <- function(M) {
  if (is.function(M)) return(dim(as.matrix(M(0L))))
  arr <- as.array(M); d <- dim(arr)
  if (length(d) >= 2L) return(d[(length(d)-1L):length(d)])
  NULL
}

# Draw observation noise v_t (T x p)  -- now SPD-safe for R
.draw_obs_noise <- function(T, p, kind = c("normal","skewnormal","exponential","cauchy"),
                            params = list(), seed = NULL) {
  kind <- match.arg(tolower(kind), c("normal","skewnormal","exponential","cauchy"))
  if (!is.null(seed)) set.seed(seed)

  if (kind == "normal") {
    R <- params$R
    if (!is.null(R)) {
      R <- as.array(R)
      if (length(dim(R)) == 2L) {
        if (!all(dim(R) == c(p, p))) stop("R must be (p,p) or (T,p,p)")
        L <- chol(.ensure_spd(R))  # SPD guard
        Z <- matrix(rnorm(T * p), nrow = T, ncol = p)
        return(Z %*% t(L))
      } else if (length(dim(R)) == 3L && all(dim(R)[1:2] == c(T, p)) && dim(R)[3] == p) {
        out <- matrix(NA_real_, nrow = T, ncol = p)
        for (t in seq_len(T)) {
          Lt <- chol(.ensure_spd(R[t, , , drop = TRUE]))  # SPD guard
          out[t, ] <- as.numeric(Lt %*% rnorm(p))
        }
        return(out)
      } else stop("R must be (p,p) or (T,p,p)")
    }
    sigma <- params$sigma; if (is.null(sigma)) sigma <- 1
    sigma <- as.numeric(sigma)
    if (length(sigma) == 1L) sigma <- rep.int(sigma, p)
    if (length(sigma) != p) stop("sigma must be scalar or length p")
    return(matrix(rnorm(T * p, sd = rep(sigma, each = T)), nrow = T, ncol = p))
  }

  if (kind == "skewnormal") {
    alpha <- params$alpha; if (is.null(alpha)) alpha <- 0
    scale <- params$scale; if (is.null(scale)) scale <- 1
    loc   <- params$loc;   if (is.null(loc))   loc   <- 0
    alpha <- rep_len(as.numeric(alpha), p)
    scale <- rep_len(as.numeric(scale), p)
    loc   <- rep_len(as.numeric(loc),   p)
    out <- matrix(NA_real_, nrow = T, ncol = p)
    for (j in seq_len(p)) out[, j] <- rsn(T, xi = loc[j], omega = scale[j], alpha = alpha[j])
    return(out)
  }

  if (kind == "exponential") {
    scale <- params$scale; if (is.null(scale)) scale <- 1
    scale <- rep_len(as.numeric(scale), p)
    out <- matrix(NA_real_, nrow = T, ncol = p)
    for (j in seq_len(p)) out[, j] <- rexp(T, rate = 1 / scale[j])
    return(out)
  }

  if (kind == "cauchy") {
    scale <- params$scale; if (is.null(scale)) scale <- 1
    loc   <- params$loc;   if (is.null(loc))   loc   <- 0
    scale <- rep_len(as.numeric(scale), p)
    loc   <- rep_len(as.numeric(loc),   p)
    out <- matrix(NA_real_, nrow = T, ncol = p)
    for (j in seq_len(p)) out[, j] <- rcauchy(T, location = loc[j], scale = scale[j])
    return(out)
  }

  stop("Unknown observation noise kind")
}

# Quantiles of observation noise q_v(p) returned as (T x p x r)
.obs_noise_quantiles <- function(T, p, kind = c("normal","skewnormal","exponential","cauchy"),
                                 params = list(), probs) {
  kind <- match.arg(tolower(kind), c("normal","skewnormal","exponential","cauchy"))
  r <- length(probs)
  out <- array(NA_real_, dim = c(T, p, r))

  if (kind == "normal") {
    R <- params$R
    if (!is.null(R)) {
      R <- as.array(R)
      z <- qnorm(probs)
      if (length(dim(R)) == 2L && all(dim(R) == c(p, p))) {
        sd <- sqrt(pmax(0, diag(.ensure_spd(R))))
        for (k in seq_len(r)) out[, , k] <- matrix(sd * z[k], nrow = T, ncol = p, byrow = TRUE)
        return(out)
      } else if (length(dim(R)) == 3L && all(dim(R)[1:2] == c(T, p)) && dim(R)[3] == p) {
        sd_t <- matrix(NA_real_, nrow = T, ncol = p)
        for (t in seq_len(T)) {
          Rt <- .ensure_spd(R[t, , , drop = TRUE])
          sd_t[t, ] <- sqrt(pmax(0, diag(Rt)))
        }
        z <- qnorm(probs)
        for (k in seq_len(r)) out[, , k] <- sd_t * z[k]
        return(out)
      } else stop("R must be (p,p) or (T,p,p)")
    }
    sigma <- params$sigma; if (is.null(sigma)) sigma <- 1
    sigma <- rep_len(as.numeric(sigma), p)
    z <- qnorm(probs)
    for (k in seq_len(r)) out[, , k] <- matrix(sigma * z[k], nrow = T, ncol = p, byrow = TRUE)
    return(out)
  }

  if (kind == "skewnormal") {
    alpha <- params$alpha; if (is.null(alpha)) alpha <- 0
    scale <- params$scale; if (is.null(scale)) scale <- 1
    loc   <- params$loc;   if (is.null(loc))   loc   <- 0
    alpha <- rep_len(as.numeric(alpha), p)
    scale <- rep_len(as.numeric(scale), p)
    loc   <- rep_len(as.numeric(loc),   p)
    for (j in seq_len(p)) {
      out[, j, ] <- matrix(qsn(probs, xi = loc[j], omega = scale[j], alpha = alpha[j]),
                           nrow = T, ncol = r, byrow = TRUE)
    }
    return(out)
  }

  if (kind == "exponential") {
    scale <- params$scale; if (is.null(scale)) scale <- 1
    scale <- rep_len(as.numeric(scale), p)
    for (j in seq_len(p)) {
      out[, j, ] <- matrix(qexp(probs, rate = 1 / scale[j]),
                           nrow = T, ncol = r, byrow = TRUE)
    }
    return(out)
  }

  if (kind == "cauchy") {
    scale <- params$scale; if (is.null(scale)) scale <- 1
    loc   <- params$loc;   if (is.null(loc))   loc   <- 0
    scale <- rep_len(as.numeric(scale), p)
    loc   <- rep_len(as.numeric(loc),   p)
    for (j in seq_len(p)) {
      out[, j, ] <- matrix(qcauchy(probs, location = loc[j], scale = scale[j]),
                           nrow = T, ncol = r, byrow = TRUE)
    }
    return(out)
  }

  stop("Unknown observation noise kind")
}

# -------------------- main simulator --------------------

#' Simulate linear state-space with flexible observation noise
simulate_state_space <- function(
  T,
  F,
  G,
  W,
  x0_mean = NULL,
  x0_cov  = NULL,
  obs_noise = c("normal","skewnormal","exponential","cauchy"),
  obs_params = list(),
  probs = NULL,
  seed = NULL
) {
  obs_noise <- match.arg(tolower(obs_noise),
                         c("normal","skewnormal","exponential","cauchy"))
  if (!is.null(seed)) set.seed(seed)

  F_shape <- .peek_shape(F)
  G_shape <- .peek_shape(G)
  if (is.null(F_shape) || is.null(G_shape)) stop("Cannot infer shapes of F and G")
  p <- F_shape[1]; m_from_F <- F_shape[2]
  m_from_G <- G_shape[1]
  if (m_from_F != m_from_G) stop(sprintf("Inconsistent state dim: F m=%d, G m=%d",
                                         m_from_F, m_from_G))
  m <- m_from_F

  Ft <- .as_timevarying(F, T, c(p, m), "F")
  Gt <- .as_timevarying(G, T, c(m, m), "G")
  Wt <- .as_timevarying(W, T, c(m, m), "W")

  if (is.null(x0_mean)) x0_mean <- rep(0, m)
  if (is.null(x0_cov))  x0_cov  <- matrix(0, m, m)
  x0_mean <- as.numeric(x0_mean)
  x0_cov  <- as.matrix(x0_cov)
  if (!all(dim(x0_cov) == c(m, m))) stop("x0_cov must be (m x m)")

  # sample x0
  if (all(abs(x0_cov) < .Machine$double.eps)) {
    x_prev <- x0_mean
  } else {
    x_prev <- as.numeric(rmvnorm(1, mean = x0_mean, sigma = .ensure_spd(x0_cov)))
  }

  # simulate states (SPD guard on W_t)
  X <- matrix(NA_real_, nrow = T, ncol = m)
  for (t in seq_len(T)) {
    Wt_slice <- .ensure_spd(Wt[t, , , drop = TRUE])
    Lw <- chol(Wt_slice)
    w_t <- as.numeric(Lw %*% rnorm(m))
    x_t <- Gt[t, , , drop = TRUE] %*% x_prev + w_t
    X[t, ] <- x_t
    x_prev <- x_t
  }

  # observations
  V <- .draw_obs_noise(T = T, p = p, kind = obs_noise, params = obs_params)
  Y_mean <- matrix(NA_real_, nrow = T, ncol = p)
  for (t in seq_len(T)) Y_mean[t, ] <- as.numeric(Ft[t, , , drop = TRUE] %*% X[t, ])
  Y <- Y_mean + V

  # conditional quantiles of y_t | x_t
  qY <- NULL
  if (!is.null(probs) && length(probs) > 0L) {
    qv <- .obs_noise_quantiles(T = T, p = p, kind = obs_noise,
                               params = obs_params, probs = probs)  # (T x p x r)
    qY <- array(NA_real_, dim = dim(qv))
    for (k in seq_len(length(probs))) qY[, , k] <- Y_mean + qv[, , k]
  }

  list(x = X, y = Y, qy = qY)
}


In [ ]:
library(ggplot2)
library(tidyr)

# Better-colors, no ribbon
plot_y_with_quantiles <- function(res, title = "y_t with quantiles", probs = NULL) {
  stopifnot(!is.null(res$y))
  Tn <- nrow(res$y); p <- ncol(res$y)
  stopifnot(p >= 1L)

  # Observed series (black)
  df_y <- data.frame(
    t = rep(seq_len(Tn), times = p),
    j = rep(paste0("y[", seq_len(p), "]"), each = Tn),
    y = as.numeric(res$y),
    stringsAsFactors = FALSE
  )

  g <- ggplot() +
    geom_line(data = df_y, aes(x = t, y = y),
              linewidth = 0.7, color = "black") +
    facet_wrap(~ j, scales = "free_y") +
    labs(x = "t", y = "value", title = title) +
    theme_minimal(base_size = 12)

  # Quantile lines (no band)
  if (!is.null(res$qy)) {
    stopifnot(length(dim(res$qy)) == 3L)
    Tn <- dim(res$qy)[1]; p <- dim(res$qy)[2]; r <- dim(res$qy)[3]

    df_q <- data.frame(
      t = rep(seq_len(Tn), times = p * r),
      j = rep(paste0("y[", seq_len(p), "]"), each = Tn * r),
      k = rep(seq_len(r), each = Tn, times = p),
      q = as.numeric(res$qy),
      stringsAsFactors = FALSE
    )

    # Label quantiles for legend
    if (!is.null(probs) && length(probs) == r) {
      # Use p values in legend (e.g., "p=0.05")
      probs_lab <- paste0("p=", format(probs, trim = TRUE))
      df_q$qlab <- probs_lab[df_q$k]
      # Find median index by nearest to 0.5
      mid_idx <- which.min(abs(probs - 0.5))
    } else {
      # Fallback: ordinal labels
      df_q$qlab <- paste0("q", df_q$k)
      mid_idx <- (r + 1L) %/% 2L  # middle in order if r is odd
    }

    # Aesthetics: highlight median, gradient for others
    df_q$is_mid <- df_q$k == mid_idx

    # Build palette: orange for median, blue gradient for others
    # Distance-from-median ranking:
    dist_from_mid <- abs(seq_len(r) - mid_idx)
    ord <- order(dist_from_mid, seq_len(r))
    # Blue gradient for non-median
    blues <- grDevices::colorRampPalette(c("#020797f2", "#730000ff"))(max(1, r - 1))
    col_map <- setNames(rep(NA_character_, r), if (!is.null(probs)) probs_lab else paste0("q", seq_len(r)))
    if (r >= 1) {
      if (r >= 2) {
        non_mid <- setdiff(seq_len(r), mid_idx)
        # assign by increasing distance so closer → lighter
        non_mid_by_dist <- non_mid[order(dist_from_mid[non_mid], non_mid)]
        col_map[(if (!is.null(probs)) probs_lab else paste0("q", seq_len(r)))[non_mid_by_dist]] <- blues
      }
      # median color
      col_map[(if (!is.null(probs)) probs_lab else paste0("q", seq_len(r)))[mid_idx]] <- "#E69F00" # orange
    }

    # Sizes/linetypes
    df_q$size <- ifelse(df_q$is_mid, 0.9, 0.5)
    df_q$lt   <- ifelse(df_q$is_mid, "solid", "solid")

    g <- g +
      geom_line(
        data = df_q,
        aes(x = t, y = q, color = qlab, linetype = lt, group = interaction(j, qlab)),
        linewidth = df_q$size,
        show.legend = TRUE
      ) +
      scale_color_manual(values = col_map, guide = guide_legend(title = "Quantiles")) +
      scale_linetype_identity()
  }

  g
}


In [ ]:
# --- Exponential observation noise demo ---

# assumes simulate_state_space(), .ensure_spd(), etc. and
# plot_y_with_quantiles() are already defined & loaded

set.seed(123)
T  <- 10000
m  <- 2
p  <- 1
G  <- matrix(c(0.99, 0.89,
               0.0, 0.89), nrow = m, byrow = TRUE)
W  <- diag(c(0.05, 0.02))
F  <- matrix(c(1.0, 0.01), nrow = p)

# Exponential noise parameters
exp_scale <- 1  # mean = scale, median = scale*log(2)

res_exp <- simulate_state_space(
  T = T, F = F, G = G, W = W,
  obs_noise  = "exponential",
  obs_params = list(scale = exp_scale),
  probs = c(0.1, 0.5, 0.9),  # choose the bands you want to see
  seed = 42
)

p_exp <- plot_y_with_quantiles(
  res_exp,
  title = "Exponential observation noise",
  probs = c(0.1, 0.5, 0.9)
)
print(p_exp)


### --- Evaluate ---

In [ ]:
## ── 0) Point at your package root (where DESCRIPTION lives)
pkg <- "~/code/exdqlm"   # adjust
stopifnot(file.exists(file.path(pkg, "DESCRIPTION")))

## ── 1) If already loaded, unload
if ("exdqlm" %in% loadedNamespaces()) {
  detach("package:exdqlm", unload = TRUE, character.only = TRUE)
  unloadNamespace("exdqlm")
}

## ── 2) Regenerate Rcpp glue
Rcpp::compileAttributes(pkgdir = pkg)

## ── 3) Sync roxygen docs/NAMESPACE
devtools::document(pkg)

## ── 4) Load dev package
devtools::load_all(pkg)

## ── 5) Sanity check C++ KF symbol
has_cpp_bridge <- exists("update_theta_bridge")
has_cpp_cpp    <- exists("update_theta_cpp")
stopifnot(has_cpp_bridge || has_cpp_cpp)

## ── ELBO options (common)
options(
  exdqlm.use_cpp_kf       = TRUE,   # toggle below for comparisons
  exdqlm.compute_elbo     = TRUE,
  exdqlm.tol_elbo         = 1e-6,
  exdqlm.use_cpp_samplers = FALSE,
  exdqlm.use_cpp_postpred = FALSE
)

## ── DATA: use your simulated exponential-noise series ------------------------
## If `res_exp` already exists, we'll reuse it; otherwise, generate it quickly:
if (!exists("res_exp")) {
  # assumes your simulate_state_space() + helpers are sourced
  set.seed(123)
  T  <- 10000; m <- 2; p <- 1
  G  <- matrix(c(0.9,0.1, 0.0,0.8), nrow = m, byrow = TRUE)
  W  <- diag(c(0.05, 0.02))
  Fm <- matrix(c(1.0, 0.3), nrow = p)
  res_exp <- simulate_state_space(
    T = T, F = Fm, G = G, W = W,
    obs_noise  = "exponential",
    obs_params = list(scale = 10),
    probs = c(0.1, 0.5, 0.9),
    seed = 42
  )
}
y_vec <- as.numeric(res_exp$y)
TT    <- length(y_vec)

In [ ]:
## ── MODEL: match simulator's exact (F, G, W) --------------------------------
# Simulator spec (copied here for clarity)
G_true <- matrix(c(0.99, 0.89,
                   0.00, 0.89), nrow = 2, byrow = TRUE)   # 2 x 2
W_true <- diag(c(0.05, 0.02)) *0                             # 2 x 2
F_true <- c(1.0, 0.01)

# Initial state: length-2 vector; initial covariance: 2x2
m0_vec <- c(0, 0)                   # <-- m0 is a VECTOR of length m (=2)
C0_mat <- diag(1, 2)              # modest prior uncertainty

# Build a single-component model that matches the simulator
true.comp <- list(
  m0 = m0_vec,                      # length-2
  C0 = C0_mat,                      # 2 x 2
  FF = F_true,                      # 1 x 2  (observational matrix)
  GG = G_true,                      # 2 x 2  (state evolution)
  W  = W_true                       # 2 x 2  (evolution covariance)
)
class(true.comp) <- "exdqlm"        # keep class consistent

# If your exdqlm expects combined models, keep as a single block:
model.w.reg <- true.comp

## ── Ensure dim.df matches total state dimension m0 --------------------------
get_m0 <- function(mod) {
  if (!is.null(mod$GG)) return(nrow(mod$GG))   # total state dimension m
  if (!is.null(mod$m))  return(mod$m)
  stop("Can't infer m0 (state dimension) from model; check structure.")
}
m_dim <- get_m0(model.w.reg)                   # should be 2 here

## Single discount block covering all states (sum == state dimension)
dim_vec <- c(m_dim)                            # c(2)
df_vec  <- c(0.999)                           # tune as needed

## ── Hyper/spec --------------------------------------------------------------
p0      <- 0.9
sig0    <- 0.01
gam0    <- 0
tol_val <- 0.01


In [ ]:
## ── Helpers (same as your originals) ----------------------------------------
theta_entropy_recalc <- function(th) {
  p  <- nrow(th$sm)
  TT <- dim(th$sC)[3]
  0.5 * sum(vapply(seq_len(TT), function(t) {
    p * (1 + log(2*pi)) + determinant(th$sC[,,t], logarithm = TRUE)$modulus[1]
  }, numeric(1)))
}
monotone_up_to <- function(x, tol_drop = 1e-4) {
  if (length(x) < 2) return(TRUE)
  min(diff(x)) >= -abs(tol_drop)
}
last_elbo <- function(fit) {
  if (!is.null(fit$diagnostics$elbo)) tail(fit$diagnostics$elbo, 1)[[1]] else NA_real_
}
gs_sanity <- function(fit, p0) {
  L <- L.fn(p0); U <- U.fn(p0)
  g  <- tail(fit$seq.gamma, 1)
  s  <- tail(fit$seq.sigma, 1)
  stopifnot(is.finite(g), is.finite(s), s > 0, g >= L - 1e-8, g <= U + 1e-8)
  invisible(list(gamma_final = g, sigma_final = s, L = L, U = U))
}
print_elbos <- function(tag, Rfit, Cfit, tol = 1e-2) {
  eR <- last_elbo(Rfit); eC <- last_elbo(Cfit)
  cat(sprintf("[%s] final ELBO — R: %.6f | C++: %.6f | diff: %.3e\n", tag, eR, eC, eC - eR))
  stopifnot(isTRUE(all.equal(eR, eC, tol = tol)))
}
check_theta_equality <- function(Rfit, Cfit) {
  stopifnot(
    isTRUE(all.equal(Rfit$theta.out$exps,  Cfit$theta.out$exps,  tol = 1e-6, check.attributes = FALSE)),
    isTRUE(all.equal(Rfit$theta.out$vars,  Cfit$theta.out$vars,  tol = 1e-6, check.attributes = FALSE)),
    isTRUE(all.equal(Rfit$theta.out$exps2, Cfit$theta.out$exps2, tol = 1e-6, check.attributes = FALSE))
  )
}
check_theta_entropy <- function(Cfit) {
  if (!is.null(Cfit$theta.out$elbo_theta)) {
    etheta_C  <- Cfit$theta.out$elbo_theta
    etheta_re <- theta_entropy_recalc(Cfit$theta.out)
    stopifnot(isTRUE(all.equal(etheta_C, etheta_re, tol = 1e-8)))
  }
}

In [ ]:
model.w.reg

In [ ]:
# ## ── A) ISVB: R-KF vs C++-KF -----------------------------------------------
# set.seed(1)
# options(exdqlm.use_cpp_kf = FALSE)
# fit_IS_R <- exdqlmISVB(
#   y        = y_vec,
#   p0       = p0,
#   model    = model.w.reg,
#   df       = df_vec,
#   dim.df   = dim_vec,
#   sig.init = sig0,
#   gam.init = gam0,
#   tol      = tol_val,
#   verbose  = TRUE
# )

# set.seed(1)
# options(exdqlm.use_cpp_kf = TRUE)
# fit_IS_C <- exdqlmISVB(
#   y        = y_vec,
#   p0       = p0,
#   model    = model.w.reg,
#   df       = df_vec,
#   dim.df   = dim_vec,
#   sig.init = sig0,
#   gam.init = gam0,
#   tol      = tol_val,
#   verbose  = TRUE
# )


In [ ]:
## ── B) LDVB: R-KF vs C++-KF -----------------------------------------------
set.seed(2)
options(exdqlm.use_cpp_kf = FALSE)
fit_LD_R <- exdqlmLDVB(
  y        = y_vec,
  p0       = p0,
  model    = model.w.reg,
  df       = df_vec,
  dim.df   = dim_vec,
  sig.init = sig0,
  gam.init = gam0,
  tol      = tol_val,
  verbose  = TRUE
)

set.seed(2)
options(exdqlm.use_cpp_kf = TRUE)
fit_LD_C <- exdqlmLDVB(
  y        = y_vec,
  p0       = p0,
  model    = model.w.reg,
  df       = df_vec,
  dim.df   = dim_vec,
  sig.init = sig0,
  gam.init = gam0,
  tol      = tol_val,
  verbose  = TRUE
)


In [ ]:
# check_theta_equality(fit_LD_R, fit_LD_C)
# stopifnot(!is.null(fit_LD_R$diagnostics$elbo), !is.null(fit_LD_C$diagnostics$elbo))
# stopifnot(monotone_up_to(fit_LD_R$diagnostics$elbo, tol_drop = 1e-3))
# stopifnot(monotone_up_to(fit_LD_C$diagnostics$elbo, tol_drop = 1e-3))
# check_theta_entropy(fit_LD_C)
# print_elbos("LDVB", fit_LD_R, fit_LD_C, tol = 1e-2)
# invisible(gs_sanity(fit_LD_R, p0)); invisible(gs_sanity(fit_LD_C, p0))


In [ ]:

## ── C) Compare algorithms on same backend (C++) ---------------------------
cmp_elbo <- function(fit1, fit2, name1="ISVB", name2="LDVB") {
  e1 <- tail(fit1$diagnostics$elbo, 1); e2 <- tail(fit2$diagnostics$elbo, 1)
  cat(sprintf("%s ELBO: %.6f | %s ELBO: %.6f | diff: %.3e\n", name1, e1, name2, e2, e2 - e1))
}
cmp_fit <- function(fit1, fit2, name1="ISVB", name2="LDVB") {
  s1 <- tail(fit1$map.standard.forecast.errors, 200)
  s2 <- tail(fit2$map.standard.forecast.errors, 200)
  cat(sprintf("MAP std. err (last 200) — %s: mean=%.3f sd=%.3f | %s: mean=%.3f sd=%.3f\n",
              name1, mean(s1), sd(s1), name2, mean(s2), sd(s2)))
}
cmp_elbo(fit_IS_C, fit_LD_C, "ISVB(C++)", "LDVB(C++)")
cmp_fit(fit_IS_C, fit_LD_C, "ISVB(C++)", "LDVB(C++)")
cat(sprintf(
  "ISVB(C++):  last sigma=%.4g, last gamma=%.4g | LDVB(C++): last sigma=%.4g, last gamma=%.4g\n",
  tail(fit_IS_C$seq.sigma, 1), tail(fit_IS_C$seq.gamma, 1),
  tail(fit_LD_C$seq.sigma, 1), tail(fit_LD_C$seq.gamma, 1)
))


In [ ]:

## ── Posterior predictive plots (zoom) --------------------------------------
# # Assumes your pp_plot_zoom() is already defined.
# pp_plot_zoom(
#   y        = y_vec,
#   samples  = fit_IS_C$samp.post.pred,
#   p0       = p0,
#   window   = c(max(1, TT - 50 + 1), TT),  # last 50 points
#   backtransform = identity,
#   spaghetti_n   = 40,
#   main     = "ISVB (C++) — last 50 points"
# )

pp_plot_zoom(
  y        = y_vec,
  samples  = fit_LD_C$samp.post.pred,
  p0       = p0,
  window   = c(max(1, TT - 50 + 1), TT),
  backtransform = identity,
  spaghetti_n   = 40,
  col_band = "tomato",
  main     = "LDVB (C++) — last 50 points"
)


In [ ]:
# install.packages("matrixStats") if you want the fast rowQuantiles
library(ggplot2)
library(tidyr)

# Compare "true" vs fitted p0-quantile traces
compare_true_vs_fitted <- function(
  res,                 # simulate_state_space() result with $qy
  samples,             # posterior predictive samples (T x ns or ns x T)
  p0,                  # target quantile level
  sim_probs = NULL,    # the 'probs' used when simulating res$qy (e.g., c(0.1,0.5,0.9))
  window = NULL,       # e.g., c(T-200+1, T) to zoom last 200; NULL = full
  y = NULL,            # optional observed y to add as context (vector length T)
  title = "True vs Fitted p0-quantile",
  backtransform = identity
) {
  stopifnot(!is.null(res$qy), is.numeric(p0), p0>0, p0<1)
  Tn <- dim(res$qy)[1]; p <- dim(res$qy)[2]; r <- dim(res$qy)[3]
  stopifnot(p >= 1L)  # we’ll use the first observation dimension (j=1)

  # 1) Select time window
  idx <- if (is.null(window)) seq_len(Tn) else {
    seq.int(max(1, window[1]), min(Tn, window[2]))
  }

  # 2) Get true Q_{p0}(y_t | x_t)
  # If sim_probs provided and includes p0, take exact match; otherwise choose nearest index.
  if (!is.null(sim_probs)) {
    stopifnot(length(sim_probs) == r)
    k_true <- which.min(abs(sim_probs - p0))
    if (abs(sim_probs[k_true] - p0) > 1e-12) {
      message(sprintf("Note: p0=%.6g not in sim_probs; using nearest level=%.6g.", p0, sim_probs[k_true]))
    }
  } else {
    # Fall back to nearest index if sim probs unknown
    k_true <- ceiling((r + 1)/2)  # assume middle is the "median/central" if p0=0.5
    message(sprintf("sim_probs not provided; using index k=%d as p0 trace.", k_true))
  }
  q_true <- res$qy[, 1, k_true][idx]
  q_true_bt <- backtransform(q_true)

  # 3) Fitted p0 trace from posterior predictive samples
  S <- samples
  if (nrow(S) != Tn) {
    if (ncol(S) == Tn) S <- t(S) else stop("samples must be T x ns or ns x T.")
  }
  S <- S[idx, , drop = FALSE]
  if (requireNamespace("matrixStats", quietly = TRUE)) {
    q_fit <- as.numeric(matrixStats::rowQuantiles(S, probs = p0, na.rm = TRUE))
  } else {
    q_fit <- apply(S, 1, quantile, probs = p0, na.rm = TRUE)
  }
  q_fit_bt <- backtransform(q_fit)

  # 4) Optional observed y (for context)
  if (!is.null(y)) {
    stopifnot(length(y) >= max(idx))
    y_bt <- backtransform(y[idx])
  } else {
    y_bt <- NULL
  }

  # 5) Metrics
  mae  <- mean(abs(q_fit_bt - q_true_bt))
  rmse <- sqrt(mean((q_fit_bt - q_true_bt)^2))
  corv <- suppressWarnings(cor(q_fit_bt, q_true_bt))

  # 6) Plot
  df <- data.frame(
    t = idx,
    true = q_true_bt,
    fit  = q_fit_bt,
    obs  = if (is.null(y_bt)) NA_real_ else y_bt
  )

  g <- ggplot(df, aes(x = t)) +
    { if (!is.null(y_bt)) geom_line(aes(y = obs), color = "grey75", linewidth = 0.6) } +
    geom_line(aes(y = true, color = "True p0"), linewidth = 1.0, linetype = "dashed") +
    geom_line(aes(y = fit,  color = "Fitted p0"), linewidth = 1.0) +
    scale_color_manual(values = c("True p0" = "black", "Fitted p0" = "#E69F00")) +
    labs(x = "t", y = "value",
         title = title,
         subtitle = sprintf("p0 = %.3g | MAE=%.4g, RMSE=%.4g, corr=%.3f", p0, mae, rmse, corv),
         color = NULL) +
    theme_minimal(base_size = 12) +
    theme(legend.position = "top")

  list(
    df      = df,
    mae     = mae,
    rmse    = rmse,
    cor     = corv,
    plot    = g
  )
}


In [ ]:
# # True vs fitted for ISVB(C++)
# cmp_is <- compare_true_vs_fitted(
#   res        = res_exp,
#   samples    = fit_IS_C$samp.post.pred,
#   p0         = 0.9,
#   sim_probs  = c(0.1, 0.5, 0.9),
#   window     = c(max(1, length(y_vec)-200+1), length(y_vec)),  # last 200 points
#   y          = y_vec,
#   title      = "ISVB(C++): True vs Fitted p0 (last 200)",
#   backtransform = identity
# )
# print(cmp_is$plot)
# cat(sprintf("ISVB: MAE=%.4g RMSE=%.4g corr=%.3f\n", cmp_is$mae, cmp_is$rmse, cmp_is$cor))

# True vs fitted for LDVB(C++)
cmp_ld <- compare_true_vs_fitted(
  res        = res_exp,
  samples    = fit_LD_C$samp.post.pred,
  p0         = 0.9,
  sim_probs  = c(0.1, 0.5, 0.9),
  window     = c(max(1, length(y_vec)-200+1), length(y_vec)),
  y          = y_vec,
  title      = "LDVB(C++): True vs Fitted p0 (last 200)",
  backtransform = identity
)
print(cmp_ld$plot)
cat(sprintf("LDVB: MAE=%.4g RMSE=%.4g corr=%.3f\n", cmp_ld$mae, cmp_ld$rmse, cmp_ld$cor))
